# 🛠️ Notebook 2: Library Management — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/library-management
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from dataclasses import dataclass, field
from datetime import date, timedelta
from typing import Optional, List
import itertools

@dataclass
class Book:
    isbn: str
    title: str
    author: str

class BookItem:
    _ids = itertools.count(1)
    def __init__(self, book: Book):
        self.barcode = f'BK-{next(BookItem._ids):04d}'
        self.book = book
        self.on_loan = False
    def __repr__(self):
        return f'{self.barcode}:{self.book.title}({"OUT" if self.on_loan else "IN"})'

@dataclass
class Loan:
    member_id: str
    item: BookItem
    borrowed_on: date
    due_on: date
    returned_on: Optional[date] = None

@dataclass
class Member:
    id: str
    name: str
    active_loans: list = field(default_factory=list)

class Library:
    MAX_LOANS = 3
    LOAN_DAYS = 14
    FINE_PER_DAY = 0.5

    def __init__(self):
        self.items_by_isbn = {}   # isbn -> list[BookItem]
        self.members = {}         # id -> Member

    def add_copy(self, book: Book):
        self.items_by_isbn.setdefault(book.isbn, []).append(BookItem(book))

    def register(self, m: Member):
        self.members[m.id] = m

    def borrow(self, member_id: str, isbn: str, today: date) -> Loan:
        m = self.members[member_id]
        if len(m.active_loans) >= self.MAX_LOANS:
            raise RuntimeError('loan limit reached')
        for item in self.items_by_isbn.get(isbn, []):
            if not item.on_loan:
                item.on_loan = True
                loan = Loan(member_id, item, today, today + timedelta(days=self.LOAN_DAYS))
                m.active_loans.append(loan)
                return loan
        raise RuntimeError('no copies available')

    def return_item(self, loan: Loan, today: date) -> float:
        loan.item.on_loan = False
        loan.returned_on = today
        self.members[loan.member_id].active_loans.remove(loan)
        late = (today - loan.due_on).days
        return round(max(0, late) * self.FINE_PER_DAY, 2)


## Walk-through

In [ ]:
lib = Library()
clean_code = Book('978-0132350884', 'Clean Code', 'R. Martin')
dddd      = Book('978-0321125217', 'Domain-Driven Design', 'E. Evans')
for _ in range(2): lib.add_copy(clean_code)
lib.add_copy(dddd)

ada = Member('M-1', 'Ada')
lib.register(ada)

today = date(2026, 4, 1)
loan = lib.borrow('M-1', clean_code.isbn, today)
print('borrowed ->', loan.item, 'due', loan.due_on)

# Return 20 days later (6 days late)
fine = lib.return_item(loan, date(2026, 4, 21))
print(f'fine = ${fine}')


### Edge cases exercised
- No copies available → `RuntimeError`.
- Loan limit enforced.
- Fine = 0 when returned on time.

### Extensions
- Reservations (waitlist) when all copies are out.
- Different loan durations per book type (reference vs. general).
- Member suspension after repeated overdue returns.